In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.combine import SMOTEENN
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [2]:
hotel = pd.read_csv(
    "processed_data/hotel_bookings_feature_engineered.csv"
)

In [3]:
X = hotel.drop("is_canceled", axis=1)

y = hotel["is_canceled"]

In [4]:
numerical_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumber of numerical variables:")
print(len(numerical_features))

print("\nNumerical variables:")
print(numerical_features)

print("\nNumber of categorical variables:")
print(len(categorical_features))

print("\nCategorical variables:")
print(categorical_features)


Number of numerical variables:
37

Numerical variables:
['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'is_family', 'room_changed', 'previous_cancellation_ratio', 'has_previous_booking', 'booked_through_agent', 'estimated_booking_value', 'adr_per_guest', 'has_weekend_stay', 'is_long_stay', 'has_special_requests', 'booking_was_changed', 'was_on_waiting_list', 'parking_required', 'arrival_month_number', 'arrival_month_sin', 'arrival_month_cos', 'adult_only_booking', 'long_lead_no_deposit']

Number of categorical variables:
12

Categorical variables:
['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', '

In [5]:
from sklearn.model_selection import train_test_split

X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Final proportions:
# 60% training
# 20% validation
# 20% testing

X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.25,
    random_state=42,
    stratify=y_development
)

print("\nTraining set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)


Training set shape: (52257, 49)
Validation set shape: (17420, 49)
Test set shape: (17420, 49)


In [6]:
# ============================================================
# CORRECT PIPELINE IMPORTS
# ============================================================

from sklearn.pipeline import Pipeline as SklearnPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.base import clone

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN


# ============================================================
# NUMERICAL PREPROCESSING PIPELINE
# ============================================================

numerical_transformer = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# CATEGORICAL PREPROCESSING PIPELINE
# ============================================================

categorical_transformer = SklearnPipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ============================================================
# COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)


# ============================================================
# DEFINE SAMPLING METHODS
# ============================================================

sampling_methods = {
    "No Resampling": None,

    "SMOTE": SMOTE(
        random_state=42,
        k_neighbors=5
    ),

    "ADASYN": ADASYN(
        random_state=42,
        n_neighbors=5
    ),

    "SMOTEENN": SMOTEENN(
        random_state=42
    )
}


# ============================================================
# DEFINE MODELS
# ============================================================

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model_name,
    sampling_name,
    fitted_pipeline,
    X_data,
    y_true
):

    y_pred = fitted_pipeline.predict(X_data)

    y_probability = fitted_pipeline.predict_proba(
        X_data
    )[:, 1]

    return {
        "Model": model_name,
        "Sampling Method": sampling_name,

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1 Score": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "ROC AUC": roc_auc_score(
            y_true,
            y_probability
        )
    }


# ============================================================
# TRAIN ALL MODEL AND SAMPLING COMBINATIONS
# ============================================================

validation_results = []
trained_pipelines = {}

for sampling_name, sampler in sampling_methods.items():

    for model_name, model in models.items():

        experiment_name = (
            f"{model_name} + {sampling_name}"
        )

        print("\n" + "=" * 70)
        print("Training:", experiment_name)
        print("=" * 70)

        try:

            # Clone objects so every experiment uses a fresh copy
            current_preprocessor = clone(preprocessor)
            current_model = clone(model)

            # ------------------------------------------------
            # NO RESAMPLING
            # ------------------------------------------------

            if sampler is None:

                model_pipeline = SklearnPipeline(
                    steps=[
                        (
                            "preprocessor",
                            current_preprocessor
                        ),
                        (
                            "classifier",
                            current_model
                        )
                    ]
                )

            # ------------------------------------------------
            # SMOTE, ADASYN OR SMOTEENN
            # ------------------------------------------------

            else:

                current_sampler = clone(sampler)

                model_pipeline = ImbPipeline(
                    steps=[
                        (
                            "preprocessor",
                            current_preprocessor
                        ),
                        (
                            "sampler",
                            current_sampler
                        ),
                        (
                            "classifier",
                            current_model
                        )
                    ]
                )

            # Train the pipeline
            model_pipeline.fit(
                X_train,
                y_train
            )

            # Evaluate on validation data
            result = evaluate_model(
                model_name=model_name,
                sampling_name=sampling_name,
                fitted_pipeline=model_pipeline,
                X_data=X_validation,
                y_true=y_validation
            )

            validation_results.append(result)

            trained_pipelines[
                experiment_name
            ] = model_pipeline

            print(
                "Validation Accuracy:",
                round(result["Accuracy"], 4)
            )

            print(
                "Validation Precision:",
                round(result["Precision"], 4)
            )

            print(
                "Validation Recall:",
                round(result["Recall"], 4)
            )

            print(
                "Validation F1 Score:",
                round(result["F1 Score"], 4)
            )

            print(
                "Validation ROC-AUC:",
                round(result["ROC AUC"], 4)
            )

        except Exception as error:

            print(
                "Training failed for:",
                experiment_name
            )

            print(
                "Error:",
                type(error).__name__,
                "-",
                error
            )


# ============================================================
# CREATE MODEL COMPARISON TABLE
# ============================================================

validation_results_df = pd.DataFrame(
    validation_results
)

if not validation_results_df.empty:

    validation_results_df = (
        validation_results_df
        .sort_values(
            by=[
                "ROC AUC",
                "F1 Score",
                "Recall"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\nValidation model comparison:")

    display(
        validation_results_df.round(4)
    )

else:

    print(
        "\nNo models were trained successfully."
    )


Training: Logistic Regression + No Resampling
Validation Accuracy: 0.8135
Validation Precision: 0.6967
Validation Recall: 0.5606
Validation F1 Score: 0.6213
Validation ROC-AUC: 0.866

Training: Decision Tree + No Resampling
Validation Accuracy: 0.8182
Validation Precision: 0.6861
Validation Recall: 0.6149
Validation F1 Score: 0.6485
Validation ROC-AUC: 0.8713

Training: Random Forest + No Resampling
Validation Accuracy: 0.8269
Validation Precision: 0.7872
Validation Recall: 0.5006
Validation F1 Score: 0.612
Validation ROC-AUC: 0.8891

Training: XGBoost + No Resampling
Validation Accuracy: 0.8377
Validation Precision: 0.7357
Validation Recall: 0.6322
Validation F1 Score: 0.68
Validation ROC-AUC: 0.9

Training: Logistic Regression + SMOTE
Validation Accuracy: 0.7808
Validation Precision: 0.5709
Validation Recall: 0.7915
Validation F1 Score: 0.6633
Validation ROC-AUC: 0.8641

Training: Decision Tree + SMOTE
Validation Accuracy: 0.7951
Validation Precision: 0.5973
Validation Recall: 0.763

,Model,Sampling Method,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,XGBoost,No Resampling,0.8377,0.7357,0.6322,0.6800,0.9000
1,XGBoost,ADASYN,0.8317,0.6904,0.6947,0.6925,0.8963
2,XGBoost,SMOTE,0.8309,0.6895,0.6919,0.6907,0.8961
3,Random Forest,ADASYN,0.8378,0.7090,0.6877,0.6982,0.8923
4,Random Forest,SMOTE,0.8356,0.7125,0.6665,0.6887,0.8901
5,Random Forest,No Resampling,0.8269,0.7872,0.5006,0.6120,0.8891
6,XGBoost,SMOTEENN,0.7754,0.5573,0.8580,0.6757,0.8873
7,Decision Tree,No Resampling,0.8182,0.6861,0.6149,0.6485,0.8713
8,Decision Tree,ADASYN,0.7746,0.5591,0.8209,0.6652,0.8692
9,Random Forest,SMOTEENN,0.7630,0.5423,0.8426,0.6599,0.8681


In [7]:
# ============================================================
# SAVE VALIDATION MODEL COMPARISON RESULTS
# ============================================================

import os

# Create the folder if it does not already exist
os.makedirs(
    "model_results",
    exist_ok=True
)

# Save the comparison table
validation_results_df.to_csv(
    "model_results/validation_model_comparison.csv",
    index=False
)

print(
    "Validation results saved successfully to:"
)

print(
    "model_results/validation_model_comparison.csv"
)

Validation results saved successfully to:
model_results/validation_model_comparison.csv


In [8]:
# ============================================================
# SAVE THE BEST HOTEL CANCELLATION MODEL
# ============================================================

import os
import joblib
import pandas as pd
from sklearn.base import clone

# Create output folders
os.makedirs("models", exist_ok=True)
os.makedirs("model_results", exist_ok=True)

# Check that model training produced results
if validation_results_df.empty:
    raise ValueError(
        "No successful model results are available."
    )

# Identify the best experiment
best_result = validation_results_df.iloc[0]

best_model_name = best_result["Model"]
best_sampling_name = best_result["Sampling Method"]

best_experiment_name = (
    f"{best_model_name} + {best_sampling_name}"
)

print("Selected best experiment:")
print(best_experiment_name)

# Check that the fitted pipeline exists
if best_experiment_name not in trained_pipelines:
    raise KeyError(
        f"{best_experiment_name} was not found "
        "inside trained_pipelines."
    )

# Retrieve the fitted best pipeline
best_model_pipeline = trained_pipelines[
    best_experiment_name
]

# Save validation comparison results
validation_results_df.to_csv(
    "model_results/validation_model_comparison.csv",
    index=False
)

# Save the fitted best model
model_save_path = (
    "models/best_hotel_cancellation_model.pkl"
)

joblib.dump(
    best_model_pipeline,
    model_save_path
)

print("\nModel saved successfully:")
print(model_save_path)

# Verify that the file exists
if os.path.exists(model_save_path):
    print("File verification successful.")
    print("Absolute model path:")
    print(os.path.abspath(model_save_path))
else:
    print("Model file was not created.")

Selected best experiment:
XGBoost + No Resampling

Model saved successfully:
models/best_hotel_cancellation_model.pkl
File verification successful.
Absolute model path:
C:\Users\ranpa\Desktop\MSc in DS\Machine Learning\Coursework\Question_1\models\best_hotel_cancellation_model.pkl


In [9]:
# ------------------------------------------------------------
# COMBINE TRAIN + VALIDATION
# ------------------------------------------------------------

X_train_final = pd.concat(
    [X_train, X_validation],
    axis=0
).reset_index(drop=True)

y_train_final = pd.concat(
    [y_train, y_validation],
    axis=0
).reset_index(drop=True)

print("\nFinal Training Shape:", X_train_final.shape)

# ------------------------------------------------------------
# REBUILD BEST PIPELINE
# ------------------------------------------------------------

selected_model = clone(models[best_model_name])
selected_preprocessor = clone(preprocessor)

selected_sampler = sampling_methods[best_sampling_name]

if selected_sampler is None:

    final_pipeline = SklearnPipeline(
        steps=[
            ("preprocessor", selected_preprocessor),
            ("classifier", selected_model)
        ]
    )

else:

    final_pipeline = ImbPipeline(
        steps=[
            ("preprocessor", selected_preprocessor),
            ("sampler", clone(selected_sampler)),
            ("classifier", selected_model)
        ]
    )

# ------------------------------------------------------------
# TRAIN FINAL MODEL
# ------------------------------------------------------------

print("\nTraining final model...")

final_pipeline.fit(
    X_train_final,
    y_train_final
)

print("Training completed.")

# ------------------------------------------------------------
# SAVE FINAL MODEL
# ------------------------------------------------------------

joblib.dump(
    final_pipeline,
    "models/best_hotel_cancellation_model.pkl"
)

print("Final model saved.")

# ------------------------------------------------------------
# GENERATE TEST PREDICTIONS
# ------------------------------------------------------------

y_test_pred = final_pipeline.predict(X_test)

y_test_prob = final_pipeline.predict_proba(
    X_test
)[:,1]

# ------------------------------------------------------------
# SAVE TEST PREDICTIONS
# ------------------------------------------------------------

test_predictions = pd.DataFrame(
    {
        "actual_is_canceled": y_test.reset_index(drop=True),
        "predicted_is_canceled": y_test_pred,
        "cancellation_probability": y_test_prob
    }
)

test_predictions.to_csv(
    "model_results/test_predictions.csv",
    index=False
)

print("Test predictions saved.")

# ------------------------------------------------------------
# QUICK TEST PERFORMANCE
# ------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("\n==============================")
print("FINAL TEST PERFORMANCE")
print("==============================")

print(
    "Accuracy :",
    round(
        accuracy_score(
            y_test,
            y_test_pred
        ),
        4
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_test,
            y_test_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall   :",
    round(
        recall_score(
            y_test,
            y_test_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "F1 Score :",
    round(
        f1_score(
            y_test,
            y_test_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "ROC AUC  :",
    round(
        roc_auc_score(
            y_test,
            y_test_prob
        ),
        4
    )
)

# ------------------------------------------------------------
# VERIFY ALL FILES
# ------------------------------------------------------------

print("\n==============================")
print("FILES CREATED")
print("==============================")

files = [
    "model_results/validation_model_comparison.csv",
    "model_results/test_predictions.csv",
    "models/best_hotel_cancellation_model.pkl"
]

for file in files:

    if os.path.exists(file):

        print("✓", file)

    else:

        print("✗", file)

print("\nEverything is ready for the Model Evaluation notebook.")


Final Training Shape: (69677, 49)

Training final model...
Training completed.
Final model saved.
Test predictions saved.

FINAL TEST PERFORMANCE
Accuracy : 0.8379
Precision: 0.7344
Recall   : 0.6359
F1 Score : 0.6816
ROC AUC  : 0.9023

FILES CREATED
✓ model_results/validation_model_comparison.csv
✓ model_results/test_predictions.csv
✓ models/best_hotel_cancellation_model.pkl

Everything is ready for the Model Evaluation notebook.
